# Experiment: Trend Baseline Varianten

## Hypothese

Multi-Timeframe-Trend (daily + weekly + monthly SMA crossover) soll besseres
Entry-Timing liefern als ein reiner Daily-50/200-Crossover.

## Setup
- Daten: `data/sample/watchlist_2020_2026.parquet` (29 Symbole, 2020–2026)
- Strategie: `multifactor_long_short` mit `ai_tech_core_ml_bundle.yaml`
- Signal-Vergleich: Daily-50/200 vs. Multi-Timeframe (daily + weekly + monthly)
- Backtest: 2023-01-01 → 2026-04-01, monatliches Rebalancing, Kosten an


In [ ]:
import sys
from pathlib import Path

# Repo root
REPO = Path.cwd()
while not (REPO / "src").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import warnings

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

print("Python path OK, repo:", REPO)

## 1. Daten laden

In [ ]:
from src.assembled_core.data.prices_ingest import load_eod_prices

PANEL_PATH = REPO / "data" / "sample" / "watchlist_2020_2026.parquet"
if not PANEL_PATH.exists():
    # Fall back to any available panel
    candidates = sorted((REPO / "data" / "sample").glob("*.parquet"))
    PANEL_PATH = candidates[0] if candidates else None
    print(f"Using fallback panel: {PANEL_PATH}")

prices = load_eod_prices(
    path=str(PANEL_PATH),
    start_date="2023-01-01",
    end_date="2026-04-01",
)
print(f"Panel: {prices.shape}, symbols: {prices['symbol'].nunique()}")
print(
    f"Date range: {prices['timestamp'].min().date()} → {prices['timestamp'].max().date()}"
)
prices.head(3)

## 2. Trend-Signale berechnen

In [ ]:
from src.assembled_core.signals.rules_trend import (
    generate_trend_signals_from_prices,
    compute_multi_timeframe_signal,
)

# Baseline: daily 50/200 crossover
signals_daily = generate_trend_signals_from_prices(
    prices,
    ma_fast=50,
    ma_slow=200,
)
print(f"Daily signals: {signals_daily.shape}")
print(signals_daily.head(3))

In [ ]:
# Multi-Timeframe: daily + weekly + monthly consensus
signals_mtf = compute_multi_timeframe_signal(
    prices,
    daily_fast=50,
    daily_slow=200,
    weekly_fast=10,
    weekly_slow=40,
    monthly_fast=3,
    monthly_slow=12,
)
print(f"MTF signals: {signals_mtf.shape}")
print(signals_mtf.head(3))

# Distribution of MTF signal
print("\nMTF signal distribution:")
print(signals_mtf["mtf_signal"].value_counts())

## 3. Signal-Qualität: IC-Analyse

In [ ]:
# Compute forward returns for IC analysis
def add_forward_return(df, n_days=21):
    """Add n-day forward return per symbol."""
    df = df.sort_values(["symbol", "timestamp"]).copy()
    df["fwd_return"] = df.groupby("symbol")["close"].transform(
        lambda s: s.shift(-n_days) / s - 1
    )
    return df


prices_with_fwd = add_forward_return(prices.copy(), n_days=21)

# Merge signals with forward returns
merged_daily = pd.merge(
    signals_daily[["timestamp", "symbol", "signal"]].rename(
        columns={"signal": "signal_daily"}
    ),
    prices_with_fwd[["timestamp", "symbol", "fwd_return"]],
    on=["timestamp", "symbol"],
    how="inner",
).dropna()

merged_mtf = pd.merge(
    signals_mtf[["timestamp", "symbol", "mtf_signal", "mtf_score"]],
    prices_with_fwd[["timestamp", "symbol", "fwd_return"]],
    on=["timestamp", "symbol"],
    how="inner",
).dropna()


# Cross-sectional IC per month
def compute_monthly_ic(merged_df, signal_col):
    merged_df = merged_df.copy()
    merged_df["month"] = merged_df["timestamp"].dt.to_period("M")
    ic = (
        merged_df.groupby("month")
        .apply(
            lambda g: g[signal_col].corr(g["fwd_return"]),
            include_groups=False,
        )
        .dropna()
    )
    return ic


ic_daily = compute_monthly_ic(merged_daily, "signal_daily")
ic_mtf = compute_monthly_ic(merged_mtf, "mtf_signal")

print(
    f"Daily IC:  mean={ic_daily.mean():.4f}  IR={ic_daily.mean() / ic_daily.std():.3f}  n={len(ic_daily)}"
)
print(
    f"MTF IC:    mean={ic_mtf.mean():.4f}   IR={ic_mtf.mean() / ic_mtf.std():.3f}  n={len(ic_mtf)}"
)

In [ ]:
# Plot IC comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

ic_daily.plot.bar(
    ax=axes[0], color=["green" if x > 0 else "red" for x in ic_daily], alpha=0.7
)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].axhline(
    ic_daily.mean(),
    color="blue",
    linestyle="--",
    linewidth=1.2,
    label=f"Mean IC={ic_daily.mean():.4f}",
)
axes[0].set_title("Monthly IC — Daily 50/200 Signal")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Cross-sectional IC")
axes[0].legend(fontsize=8)
axes[0].tick_params(axis="x", rotation=45, labelsize=6)

ic_mtf.plot.bar(
    ax=axes[1], color=["green" if x > 0 else "red" for x in ic_mtf], alpha=0.7
)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].axhline(
    ic_mtf.mean(),
    color="orange",
    linestyle="--",
    linewidth=1.2,
    label=f"Mean IC={ic_mtf.mean():.4f}",
)
axes[1].set_title("Monthly IC — Multi-Timeframe Signal")
axes[1].set_xlabel("Month")
axes[1].legend(fontsize=8)
axes[1].tick_params(axis="x", rotation=45, labelsize=6)

fig.suptitle(
    "IC Comparison: Daily vs. Multi-Timeframe Trend Signal",
    fontsize=12,
    fontweight="bold",
)
fig.tight_layout()
plt.savefig(REPO / "output" / "ic_comparison_trend.png", dpi=120, bbox_inches="tight")
plt.show()
print("[OK] Saved: output/ic_comparison_trend.png")

## 4. Breakout-Signal vergleichen

In [ ]:
# ARCHIVIERT 2026-08-17 (Audit 6.4 Tranche 1): breakout_signal liegt unter
# archive/orphaned_code_2026-08-17/ (Bindestriche - kein importierbares Paket).
# Aktiver Load ueber den Archivpfad — REPO-verankert, weil Jupyter mit
# CWD = Notebook-Verzeichnis laeuft, nicht Repo-Root (E-169/E-146):
import importlib.util as _ilu

_spec = _ilu.spec_from_file_location(
    "breakout_signal",
    str(
        REPO / "archive" / "orphaned_code_2026-08-17" / "signals" / "breakout_signal.py"
    ),
)
_bs = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_bs)
compute_breakout_signals_panel = _bs.compute_breakout_signals_panel

prices_with_breakout = compute_breakout_signals_panel(
    prices.rename(columns={"timestamp": "date"}),
    date_col="date",
    channel_days=20,
    confirm_days=3,
    use_atr_filter=True,
)
print(f"Breakout panel: {prices_with_breakout.shape}")
print("Breakout direction distribution:")
print(prices_with_breakout["breakout_direction"].value_counts())

In [ ]:
# IC for breakout signal
prices_with_breakout_ts = prices_with_breakout.rename(columns={"date": "timestamp"})
prices_with_fwd_bt = add_forward_return(
    prices_with_breakout_ts[["timestamp", "symbol", "close", "breakout_score"]],
    n_days=21,
)

merged_bt = prices_with_fwd_bt.dropna(subset=["breakout_score", "fwd_return"])
ic_breakout = compute_monthly_ic(merged_bt, "breakout_score")

print(
    f"Daily IC:    mean={ic_daily.mean():.4f}  IR={ic_daily.mean() / ic_daily.std():.3f}"
)
print(f"MTF IC:      mean={ic_mtf.mean():.4f}   IR={ic_mtf.mean() / ic_mtf.std():.3f}")
print(
    f"Breakout IC: mean={ic_breakout.mean():.4f}  IR={ic_breakout.mean() / ic_breakout.std():.3f}"
)

## 5. Fazit

Trage hier die Ergebnisse ein:

| Signal | Mean IC | IR |
|--------|---------|----|
| Daily 50/200 | *aus Cell 4* | *aus Cell 4* |
| Multi-Timeframe | *aus Cell 4* | *aus Cell 4* |
| Breakout (Donchian) | *aus Cell 7* | *aus Cell 7* |

**Hypothese bestätigt?** Trage hier das Ergebnis ein.

## Nächste Schritte

- Kombination aller drei Signale als Ensemble-Score
- A/B-Backtest mit `run_backtest_strategy.py` gegen `ai_tech_core_ml_bundle.yaml`
- Walk-Forward-Validierung mit `src/assembled_core/qa/walk_forward_optuna.py`
